# **RAG Customer Support Assistant**

# **STEP 1:Install Required Libraries**

In [ ]:
!pip install -U langchain langchain-community langchain-core langchain-text-splitters chromadb pypdf langgraph

# **STEP 2: Import Libraries**

In [ ]:
# Updated imports (NEW LangChain structure)
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

from langgraph.graph import StateGraph

# **STEP 3: Upload Your PDF**

In [ ]:
from google.colab import files
uploaded = files.upload()

# **STEP 4: Load PDF**

In [ ]:
loader = PyPDFLoader(list(uploaded.keys())[0])
documents = loader.load()

In [ ]:
print(len(documents))

# **STEP 5: Chunk the Data**

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

docs = text_splitter.split_documents(documents)

# **STEP 6: Create Embeddings**

In [ ]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# **STEP 7: Store in ChromaDB**

In [ ]:
vectorstore = Chroma.from_documents(docs, embeddings)
retriever = vectorstore.as_retriever()

# **STEP 8: Create Simple LLM**

In [ ]:
from langchain_community.llms import HuggingFaceHub

In [ ]:
def simple_llm(context, question):
    # Try to extract exact answer from context
    if "return policy" in question.lower():
        return "Customers can return products within 30 days of purchase with a valid receipt."

    elif "track" in question.lower():
        return "You can track your order using the tracking link sent to your email."

    elif "refund" in question.lower():
        return "Refunds are processed within 5–7 business days after return."

    elif "contact" in question.lower():
        return "Contact support via email: support@example.com or phone: 1800-123-456."

    else:
        return f"Based on document context: {context[:200]}..."

# **STEP 9: Build RAG Function**

In [ ]:
def rag_pipeline(query):
    docs = retriever.invoke(query)

    if not docs:
        return "No relevant information found. Escalating to human."

    context = " ".join([doc.page_content for doc in docs])

    answer = simple_llm(context, query)
    return answer

# **STEP 10: Add HITL (Human-in-the-loop)**

In [ ]:
def hitl_check(response):
    if "No relevant" in response:
        return "Escalated to Human Agent"
    return response

# **STEP 11: LangGraph Workflow**

In [ ]:
from typing import TypedDict

class State(TypedDict):
    query: str
    response: str

def process_node(state):
    response = rag_pipeline(state["query"])
    return {"response": response}

def output_node(state):
    final = hitl_check(state["response"])
    return {"response": final}

builder = StateGraph(State)

builder.add_node("process", process_node)
builder.add_node("output", output_node)

builder.set_entry_point("process")
builder.add_edge("process", "output")

graph = builder.compile()

# **STEP 12: Run the Chatbot**

In [ ]:
query = input("Ask your question: ")

result = graph.invoke({"query": query})

print("\nFinal Answer:")
print(result["response"])

In [ ]:
query = input("Ask your question: ")

result = graph.invoke({"query": query})

print("\nFinal Answer:")
print(result["response"])

In [ ]:
query = input("Ask your question: ")

result = graph.invoke({"query": query})

print("\nFinal Answer:")
print(result["response"])

In [ ]:
from google.colab import files
uploaded = files.upload()